# 04 · Evaluation & Paper Figures

**Purpose:** Generate all evaluation metrics, diagnostic plots, and paper-ready figures
from the fitted models (M0–M3). Also includes the WPA decision-landscape analysis
and kicker FGOE leaderboard.

**Inputs:**
- `reports/xfg_success/fg_full_with_predictions.csv` (from notebook 03)
- `reports/attempt_pi/attempt_pi_oof_predictions_final.csv` (from notebook 02)
- `models/m1/m1_fg_B_logit.rds`
- `data/fg_all.csv` (for era / season trend analyses)

**Outputs (all to `reports/figures/`):**

| File | Description |
|------|-------------|
| `fig_01_calibration_by_model.png` | Calibration curves for M0–M3 |
| `fig_02_residual_by_distance.png` | Binned Brier residuals vs distance |
| `fig_03_weight_distribution.png` | IPW weight distribution histogram |
| `fig_04_weights_vs_distance.png` | IPW weights vs kick distance |
| `fig_05_rationality_density.png` | P(FG \| context) density by decision type |
| `fig_06_wpa_decision_landscape.png` | WPA landscape for FG vs Go vs Punt |
| `fig_07_era_comparison.png` | xFG by era (pre/post-2015) |
| `fig_08_kicking_evolution.png` | Seasonal make% and xFG trend |
| `fig_09_distance_distribution.png` | FG attempt distance distribution |
| `fig_10_fgoe_leaderboard.png` | Top kickers by FGOE (field goals over expected) |
| `fig_11_skill_vs_workload.png` | Kicker FGOE vs attempt volume scatter |
| `fig_12_metrics_table.csv` | Numeric metrics table |

**Requires:** notebooks 01–03 outputs.

In [1]:
# ============================================================
# 1. Parameters
# ============================================================
PROJECT_ROOT <- sub('[/\\][^/\\]*$', '', getwd())

data_dir          <- file.path(PROJECT_ROOT, 'data')
models_dir        <- file.path(PROJECT_ROOT, 'models')
final_models_dir  <- file.path(models_dir, 'final_models')
reports_dir       <- file.path(PROJECT_ROOT, 'reports')
figures_dir       <- file.path(reports_dir, 'figures')

# Evaluation window — must match notebook 03 to resolve the correct test ID file
EVAL_SEASON_MIN <- 2015L
EVAL_SEASON_MAX <- 2025L

# Canonical model artifact names from notebook 03
MODEL_FILE_M0     <- 'xfg_m0_dist_only_logit.rds'
MODEL_FILE_M1     <- 'xfg_m1_full_logit.rds'
MODEL_FILE_M2     <- 'xfg_m2_ipw_logit.rds'
MODEL_FILE_M2_POP <- 'xfg_m2_ipw_no_kicker_season_logit.rds'
MODEL_FILE_M3     <- 'xfg_m3_augmented_logit.rds'

CALIB_BINS     <- 10L      # calibration bins
RESID_BINS     <- 10L      # residual curve bins
FIG_WIDTH      <- 8.0      # inches
FIG_HEIGHT     <- 5.5      # inches
FIG_DPI        <- 300
MIN_FGOE_ATT   <- 25L      # minimum FG attempts for leaderboard

# Which model column to use as the primary "xFG" for FGOE etc.
XFG_COL    <- 'p_m2'       # IPW-corrected
AUG_COL    <- 'p_m3'       # augmented

if (!dir.exists(figures_dir)) dir.create(figures_dir, recursive = TRUE)
message('figures_dir: ', figures_dir)
message('final_models_dir: ', final_models_dir)

figures_dir: g:/Other computers/Desktop/My Files/Python/Sports Analytics Projects/Football/Kickers/NFL-Field-Goal-Kicker-Model/reports/figures

final_models_dir: g:/Other computers/Desktop/My Files/Python/Sports Analytics Projects/Football/Kickers/NFL-Field-Goal-Kicker-Model/models/final_models



In [2]:
# ============================================================
# 2. Imports
# ============================================================
dependencies <- c(
  'dplyr', 'tibble', 'tidyr', 'readr', 'stringr', 'purrr',
  'ggplot2', 'ggrepel', 'scales', 'patchwork', 'pROC', 'glmmTMB'
)
installed <- rownames(installed.packages())
for (pkg in dependencies) {
  if (!pkg %in% installed) install.packages(pkg)
  suppressPackageStartupMessages(library(pkg, character.only = TRUE))
}

# Minimal paper theme: no grids, light border, clean typography
theme_paper <- function(base_size = 11) {
  ggplot2::theme_minimal(base_size = base_size) +
    ggplot2::theme(
      panel.grid.major = ggplot2::element_blank(),
      panel.grid.minor = ggplot2::element_blank(),
      panel.border = ggplot2::element_rect(fill = NA, colour = 'grey75', linewidth = 0.5),
      axis.line = ggplot2::element_line(colour = 'grey45', linewidth = 0.3),
      strip.background = ggplot2::element_rect(fill = 'grey95', colour = 'grey80', linewidth = 0.4),
      strip.text = ggplot2::element_text(face = 'bold', colour = 'grey25'),
      legend.position = 'bottom',
      legend.key = ggplot2::element_blank(),
      plot.title = ggplot2::element_text(face = 'bold', size = base_size + 2),
      plot.subtitle = ggplot2::element_text(colour = 'grey35', size = base_size),
      axis.title = ggplot2::element_text(size = base_size)
    )
}

save_fig <- function(p, fname, w = FIG_WIDTH, h = FIG_HEIGHT, dpi = FIG_DPI) {
  path <- file.path(figures_dir, fname)
  ggplot2::ggsave(path, plot = p, width = w, height = h, dpi = dpi)
  message('Saved: ', path)
  invisible(path)
}

message('Libraries loaded.')

Warning message:
"package 'ggrepel' was built under R version 4.5.2"
Warning message:
"package 'patchwork' was built under R version 4.5.2"
Warning message:
"package 'pROC' was built under R version 4.5.2"
Warning message:
"package 'glmmTMB' was built under R version 4.5.2"
Libraries loaded.



In [3]:
# ============================================================
# 3. Helper Functions
# ============================================================

brier   <- function(y, p) mean((y - p)^2, na.rm = TRUE)
logloss <- function(y, p, eps = 1e-15)
  -mean(y * log(pmin(pmax(p, eps), 1-eps)) + (1-y) * log(1-pmin(pmax(p, eps), 1-eps)), na.rm=TRUE)
auc_fn  <- function(y, p) tryCatch(as.numeric(pROC::auc(y, p, quiet = TRUE)), error = function(e) NA_real_)

calc_metrics <- function(y, p, set_name) {
  tibble::tibble(
    model = set_name, n = sum(!is.na(y) & !is.na(p)),
    brier = brier(y, p), logloss = logloss(y, p), auc = auc_fn(y, p)
  )
}

# Calibration data
calib_data <- function(y, p, nbins = CALIB_BINS, label = '') {
  df <- tibble::tibble(y = y, p = p) %>% filter(!is.na(y), !is.na(p))
  df$bin <- ggplot2::cut_number(df$p, nbins)
  df %>% group_by(bin) %>%
    summarise(obs = mean(y), pred = mean(p), n = n(), .groups = 'drop') %>%
    mutate(model = label)
}

## 4. Load Data

In [4]:
preds   <- readr::read_csv(file.path(reports_dir, 'xfg_success','fg_full_with_predictions.csv'),
                            show_col_types = FALSE)
preds_pi <- readr::read_csv(file.path(reports_dir, 'attempt_pi', 'attempt_pi_oof_predictions_final.csv'),
                             show_col_types = FALSE)
fg_all   <- readr::read_csv(file.path(data_dir, 'fg_all.csv'), show_col_types = FALSE)

# Load canonical models from models/final_models
m0 <- readRDS(file.path(final_models_dir, MODEL_FILE_M0))
m1 <- readRDS(file.path(final_models_dir, MODEL_FILE_M1))
m2 <- readRDS(file.path(final_models_dir, MODEL_FILE_M2))
m2_pop <- readRDS(file.path(final_models_dir, MODEL_FILE_M2_POP))
m3 <- readRDS(file.path(final_models_dir, MODEL_FILE_M3))

message('Loaded canonical model artifacts from final_models.')

message('preds:    ', nrow(preds), ' FG rows')
message('preds_pi: ', nrow(preds_pi), ' rows | seasons ', min(preds_pi$season), '-', max(preds_pi$season))
message('fg_all:   ', nrow(fg_all), ' rows')

# Test set mask — filename mirrors notebook 03's TEST_GAME_IDS_FILE
test_game_ids_file <- file.path(data_dir, 'augmented',
  sprintf('test_fg_game_ids_%d_%d.csv', EVAL_SEASON_MIN, EVAL_SEASON_MAX))
test_game_ids <- if (file.exists(test_game_ids_file)) {
  readr::read_csv(test_game_ids_file, show_col_types = FALSE)$game_id
} else {
  warning('Test game IDs file not found: ', test_game_ids_file,
          '\nRun notebook 03 first to generate the split.')
  character(0)
}
message('Test game IDs loaded: ', length(test_game_ids), ' games from ', test_game_ids_file)

preds <- preds %>%
  mutate(
    split    = if_else(game_id %in% test_game_ids, 'test', 'train'),
    # Season-based era groupings (all data is 2015-2025)
    era      = dplyr::case_when(
      season <= 2018 ~ '2015-2018',
      season <= 2022 ~ '2019-2022',
      TRUE           ~ '2023-2025'
    ),
    fgoe_m1 = kick_made - p_m1,
    fgoe_m2 = kick_made - p_m2,
    fgoe_m3 = kick_made - p_m3
  )

# In-sample (all rows; 2015-2025 data only in this file)
preds_is <- preds
y_is     <- preds_is$kick_made

# Test subset
preds_test <- preds %>% filter(split == 'test')
y_test     <- preds_test$kick_made

message('in-sample rows: ', nrow(preds_is), ' | test rows: ', nrow(preds_test))
message('seasons: ', min(preds_is$season), '-', max(preds_is$season))

Loaded canonical model artifacts from final_models.

preds:    11548 FG rows

preds_pi: 22879 rows | seasons 2015-2025

fg_all:   137668 rows

Test game IDs loaded: 593 games from g:/Other computers/Desktop/My Files/Python/Sports Analytics Projects/Football/Kickers/NFL-Field-Goal-Kicker-Model/data/augmented/test_fg_game_ids_2015_2025.csv

in-sample rows: 11548 | test rows: 2251

seasons: 2015-2025



## 4b. Population-Level xFG (M2 without kicker random effect)

For **player attribution** (FGOE leaderboards, skill scatter), we must not use a model that
already encodes individual kicker ability. M1/M2 include `(1 | kicker_player_id:season_f)`,
so their predictions absorb each kicker's skill, pushing FGOE toward zero for elite kickers
(e.g., Brandon Aubrey). The fix: refit M2 keeping only the **venue** random effect
`(1 | stadium_id)` — venue is environmental, not skill. This gives a population baseline
that a typical kicker of any given distance/environment is expected to make, making residuals
(`kick_made − p_m2_pop`) a clean measure of individual skill.

In [5]:
# ============================================================
# 4b. Population-level xFG from saved M2-pop model
# ============================================================
if (is.null(m2_pop)) stop('M2-pop model artifact not loaded from models/final_models')

# Align key types for prediction with random effects
preds_is <- preds_is %>%
  mutate(
    stadium_id = as.character(stadium_id),
    is_ot      = as.logical(is_ot)
  )

preds_test <- preds_test %>%
  mutate(
    stadium_id = as.character(stadium_id),
    is_ot      = as.logical(is_ot)
  )

# Use plain data.frames for glmmTMB prediction robustness
preds_is_df <- as.data.frame(preds_is)
preds_test_df <- as.data.frame(preds_test)

# Population-level predictions for fair player attribution
p_m2_pop_is <- predict(m2_pop, newdata = preds_is_df, type = 'response', allow.new.levels = TRUE)
p_m2_pop_test <- predict(m2_pop, newdata = preds_test_df, type = 'response', allow.new.levels = TRUE)

preds_is <- preds_is %>%
  mutate(
    p_m2_pop = p_m2_pop_is,
    fgoe_pop = kick_made - p_m2_pop
  )

preds_test <- preds_test %>%
  mutate(
    p_m2_pop = p_m2_pop_test,
    fgoe_pop = kick_made - p_m2_pop
  )

# Sanity checks
cat('\n=== M2-pop sanity checks ===\n')
cat('Mean fgoe_pop (should be ~0):', round(mean(preds_is$fgoe_pop, na.rm = TRUE), 4), '\n')
cat('Mean p_m2_pop:', round(mean(preds_is$p_m2_pop, na.rm = TRUE), 4),
    '  vs mean p_m2:', round(mean(preds_is$p_m2, na.rm = TRUE), 4), '\n')

aubrey <- preds_is %>%
  filter(stringr::str_detect(kicker_player_name, regex('aubrey', ignore_case = TRUE))) %>%
  summarise(
    n = n(),
    fgoe_total = sum(fgoe_pop, na.rm = TRUE),
    fgoe_per_att = mean(fgoe_pop, na.rm = TRUE),
    p_m2_pop_mean = mean(p_m2_pop, na.rm = TRUE)
  )

cat('\nAubrey (fgoe_pop): n=', aubrey$n,
    ' total=', round(aubrey$fgoe_total, 2),
    ' per_att=', round(aubrey$fgoe_per_att, 4), '\n')


=== M2-pop sanity checks ===
Mean fgoe_pop (should be ~0): 0.0042 
Mean p_m2_pop: 0.8585   vs mean p_m2: 0.8607 

Aubrey (fgoe_pop): n= 125  total= 9.89  per_att= 0.0792 


## 5. Metrics Table

In [6]:
# In-sample metrics (primary)
metrics_is <- dplyr::bind_rows(
  calc_metrics(y_is, preds_is$p_m0, 'M0 (distance only)'),
  calc_metrics(y_is, preds_is$p_m1, 'M1 (full GLMM)'),
  calc_metrics(y_is, preds_is$p_m2, 'M2 (IPW-corrected)'),
  calc_metrics(y_is, preds_is$p_m3, 'M3 (augmented)')
) %>% mutate(split = 'in_sample', .before = 1)

# Test set metrics (supplemental)
metrics_test <- dplyr::bind_rows(
  calc_metrics(y_test, preds_test$p_m0, 'M0 (distance only)'),
  calc_metrics(y_test, preds_test$p_m1, 'M1 (full GLMM)'),
  calc_metrics(y_test, preds_test$p_m2, 'M2 (IPW-corrected)'),
  calc_metrics(y_test, preds_test$p_m3, 'M3 (augmented)')
) %>% mutate(split = 'test', .before = 1)

metrics <- dplyr::bind_rows(metrics_is, metrics_test)
print(metrics)
readr::write_csv(metrics, file.path(figures_dir, 'fig_12_metrics_table.csv'))


# A tibble: 8 × 6
  split     model                  n  brier logloss   auc
  <chr>     <chr>              <int>  <dbl>   <dbl> <dbl>
1 in_sample M0 (distance only) 11548 0.105    0.339 0.774
2 in_sample M1 (full GLMM)     11548 0.102    0.329 0.796
3 in_sample M2 (IPW-corrected) 11548 0.0999   0.326 0.798
4 in_sample M3 (augmented)     11548 0.102    0.332 0.791
5 test      M0 (distance only)  2251 0.101    0.332 0.762
6 test      M1 (full GLMM)      2251 0.101    0.330 0.769
7 test      M2 (IPW-corrected)  2251 0.103    0.337 0.760
8 test      M3 (augmented)      2251 0.102    0.333 0.767


## 6. Paper Figures

In [7]:
# fig_01: Calibration curves (in-sample primary) — line-only for readability
calib_df <- dplyr::bind_rows(
  calib_data(y_is, preds_is$p_m0, label = 'M0'),
  calib_data(y_is, preds_is$p_m1, label = 'M1'),
  calib_data(y_is, preds_is$p_m2, label = 'M2'),
  calib_data(y_is, preds_is$p_m3, label = 'M3')
)

p_calib <- ggplot2::ggplot(calib_df, ggplot2::aes(pred, obs, colour = model)) +
  ggplot2::geom_abline(slope = 1, intercept = 0, lty = 2, colour = 'grey50') +
  ggplot2::geom_line(linewidth = 1.0) +
  ggplot2::scale_colour_brewer(palette = 'Set1') +
  ggplot2::coord_equal(xlim = c(0, 1), ylim = c(0, 1)) +
  ggplot2::labs(
    title = 'Calibration Curves (In-Sample, 2015-2025)',
    subtitle = 'Line-only rendering to reduce visual crowding',
    x = 'Mean Predicted P(make)', y = 'Observed Make Rate',
    colour = NULL
  ) +
  theme_paper()

save_fig(p_calib, 'fig_01_calibration_by_model.png')


Saved: g:/Other computers/Desktop/My Files/Python/Sports Analytics Projects/Football/Kickers/NFL-Field-Goal-Kicker-Model/reports/figures/fig_01_calibration_by_model.png



In [8]:
# fig_02: Binned Brier residuals vs distance (in-sample)
resid_df <- preds_is %>%
  filter(!is.na(kick_made)) %>%
  mutate(dist_bin = cut(kick_distance,
                        breaks = c(-Inf, 25, 30, 35, 40, 45, 50, 55, 60, Inf),
                        labels = c('<25','26-30','31-35','36-40','41-45','46-50','51-55','56-60','>60'),
                        include.lowest = TRUE)) %>%
  tidyr::pivot_longer(c(p_m0, p_m1, p_m2, p_m3),
                      names_to = 'model', values_to = 'p') %>%
  mutate(model = toupper(stringr::str_remove(model, 'p_'))) %>%
  group_by(model, dist_bin) %>%
  summarise(brier_bin = brier(kick_made, p), n = n(), .groups = 'drop')

p_resid <- ggplot2::ggplot(resid_df, ggplot2::aes(dist_bin, brier_bin, group = model, colour = model)) +
  ggplot2::geom_line(linewidth = 0.9) +
  ggplot2::geom_point() +
  ggplot2::scale_colour_brewer(palette = 'Set1') +
  ggplot2::labs(title = 'Brier Score by Distance Band (In-Sample, 2015-2025)',
    x = 'Kick Distance (yards)', y = 'Brier Score', colour = NULL) +
  theme_paper() +
  ggplot2::theme(axis.text.x = ggplot2::element_text(angle = 30, hjust = 1))

save_fig(p_resid, 'fig_02_residual_by_distance.png')


Saved: g:/Other computers/Desktop/My Files/Python/Sports Analytics Projects/Football/Kickers/NFL-Field-Goal-Kicker-Model/reports/figures/fig_02_residual_by_distance.png



In [9]:
# fig_03: IPW weight distribution with annotated reference and sub-1 share
w_df <- preds %>%
  filter(!is.na(w_ipw_final), is.finite(w_ipw_final))

pct_lt1 <- mean(w_df$w_ipw_final < 1, na.rm = TRUE)
label_y <- max(hist(w_df$w_ipw_final, plot = FALSE, breaks = 60)$counts, na.rm = TRUE) * 0.92

p_wdist <- ggplot2::ggplot(w_df, ggplot2::aes(w_ipw_final)) +
  ggplot2::geom_histogram(bins = 60, fill = '#2166AC', colour = NA, alpha = 0.8) +
  ggplot2::geom_vline(xintercept = 1, lty = 2, colour = 'grey40') +
  ggplot2::annotate('text', x = 1.03, y = label_y, label = 'w = 1 reference',
                    hjust = 0, vjust = 1, colour = 'grey25', size = 3.3) +
  ggplot2::annotate('label', x = 3.9, y = label_y,
                    label = paste0('Kicks with w < 1: ', scales::percent(pct_lt1, accuracy = 0.1)),
                    fill = 'white', colour = 'grey20', size = 3.2, linewidth = 0.2) +
  ggplot2::coord_cartesian(xlim = c(0, 6)) +
  ggplot2::labs(
    title = 'IPW Final Weight Distribution',
    subtitle = 'Dashed line marks neutral weight (w = 1)',
    x = 'w_ipw_final', y = 'Count'
  ) +
  theme_paper()

save_fig(p_wdist, 'fig_03_weight_distribution.png')

Saved: g:/Other computers/Desktop/My Files/Python/Sports Analytics Projects/Football/Kickers/NFL-Field-Goal-Kicker-Model/reports/figures/fig_03_weight_distribution.png



In [10]:
# fig_04: IPW weights vs kick distance
p_wvsdist <- preds %>%
  filter(!is.na(w_ipw_final), is.finite(w_ipw_final)) %>%
  ggplot2::ggplot(ggplot2::aes(kick_distance, w_ipw_final)) +
  ggplot2::geom_point(alpha = 0.15, size = 0.7, colour = '#2166AC') +
  ggplot2::geom_smooth(method = 'loess', colour = '#D6604D', se = FALSE, linewidth = 1.2) +
  ggplot2::geom_hline(yintercept = 1, lty = 2, colour = 'grey40') +
  ggplot2::scale_y_continuous(limits = c(0, 8)) +
  ggplot2::labs(title = 'IPW Weights vs. Kick Distance',
    x = 'Kick Distance (yards)', y = 'w_ipw_final') +
  theme_paper()

save_fig(p_wvsdist, 'fig_04_weights_vs_distance.png')

`geom_smooth()` using formula = 'y ~ x'
Warning message:
"Removed 20 rows containing non-finite outside the scale range
(`stat_smooth()`)."
Warning message:
"Removed 20 rows containing missing values or values outside the scale range
(`geom_point()`)."
Warning message:
"Removed 1 row containing missing values or values outside the scale range
(`geom_smooth()`)."
Saved: g:/Other computers/Desktop/My Files/Python/Sports Analytics Projects/Football/Kickers/NFL-Field-Goal-Kicker-Model/reports/figures/fig_04_weights_vs_distance.png



In [11]:
# fig_05: Propensity density by decision type
rationality_df <- preds_pi %>%
  filter(!is.na(p_hat_multinom)) %>%
  mutate(decision_type = if_else(attempt_fg == 1, 'FG Attempted', 'Not Attempted'))

p_rat <- ggplot2::ggplot(rationality_df, ggplot2::aes(p_hat_multinom, fill = decision_type)) +
  ggplot2::geom_density(alpha = 0.55, colour = NA) +
  ggplot2::scale_fill_manual(values = c('FG Attempted' = '#2166AC', 'Not Attempted' = '#D6604D')) +
  ggplot2::labs(title = 'Propensity Density: FG vs. Non-FG Decisions',
    x = expression(hat(pi)(x) * ' — P(FG | context)'),
    y = 'Density', fill = NULL) +
  theme_paper()

save_fig(p_rat, 'fig_05_rationality_density.png')

Saved: g:/Other computers/Desktop/My Files/Python/Sports Analytics Projects/Football/Kickers/NFL-Field-Goal-Kicker-Model/reports/figures/fig_05_rationality_density.png



In [12]:
# fig_06: WPA decision landscape
# Use fg_all with leverage + WPA columns and FG/Punt/Go decisions
# Filter to 4th-down decisions with sufficient overlap
wpa_df <- fg_all %>%
  filter(
    is_pat == 0L,
    play_type_original %in% c('field_goal', 'pass', 'run', 'punt'),
    !is.na(kick_distance), kick_distance <= 70,
    !is.na(wpa),
    !is.na(leverage_rules)
  ) %>%
  mutate(
    decision_type = dplyr::case_when(
      play_type_original == 'field_goal' ~ 'FG',
      play_type_original == 'punt'       ~ 'Punt',
      TRUE                               ~ 'Go'
    )
  )

p_wpa <- ggplot2::ggplot(wpa_df,
    ggplot2::aes(kick_distance, wpa, colour = decision_type)) +
  ggplot2::geom_hline(yintercept = 0, colour = 'grey60', lty = 2) +
  ggplot2::geom_smooth(method = 'loess', se = TRUE, alpha = 0.12, linewidth = 1.1) +
  ggplot2::scale_colour_manual(values = c('FG' = '#2166AC', 'Punt' = '#4DAC26', 'Go' = '#D6604D')) +
  ggplot2::labs(title = 'Win Probability Added by Decision Type',
    x = 'Kick Distance (yards)', y = 'WPA', colour = NULL) +
  theme_paper()

save_fig(p_wpa, 'fig_06_wpa_decision_landscape.png')

`geom_smooth()` using formula = 'y ~ x'
Saved: g:/Other computers/Desktop/My Files/Python/Sports Analytics Projects/Football/Kickers/NFL-Field-Goal-Kicker-Model/reports/figures/fig_06_wpa_decision_landscape.png



In [13]:
# fig_07: The Modern Kicking Era 2015-2025 — four-panel season-level evolution
season_trends <- preds_is %>%
  filter(!is.na(kick_made), !is.na(kick_distance), !is.na(p_m2)) %>%
  group_by(season) %>%
  summarise(
    avg_distance   = mean(kick_distance),
    pct_50plus     = mean(kick_distance >= 50),
    make_pct_50p   = mean(kick_made[kick_distance >= 50], na.rm = TRUE),
    xfg_mean       = mean(p_m2),
    n              = n(),
    .groups        = 'drop'
  )

p_a <- ggplot2::ggplot(season_trends, ggplot2::aes(season, avg_distance)) +
  ggplot2::geom_line(colour = '#2166AC', linewidth = 0.9) +
  ggplot2::geom_point(colour = '#2166AC', size = 1.8) +
  ggplot2::scale_y_continuous(limits = c(35, 45)) +
  ggplot2::labs(title = 'Avg Kick Distance', x = NULL, y = 'Yards') +
  theme_paper()

p_b <- ggplot2::ggplot(season_trends, ggplot2::aes(season, pct_50plus)) +
  ggplot2::geom_line(colour = '#D6604D', linewidth = 0.9) +
  ggplot2::geom_point(colour = '#D6604D', size = 1.8) +
  ggplot2::scale_y_continuous(limits = c(0.10, 0.35), labels = scales::percent_format(accuracy = 1)) +
  ggplot2::labs(title = '% Attempts >= 50 Yards', x = NULL, y = NULL) +
  theme_paper()

p_c <- ggplot2::ggplot(season_trends, ggplot2::aes(season, make_pct_50p)) +
  ggplot2::geom_line(colour = '#4DAC26', linewidth = 0.9) +
  ggplot2::geom_point(colour = '#4DAC26', size = 1.8) +
  ggplot2::scale_y_continuous(limits = c(0.50, 0.85), labels = scales::percent_format(accuracy = 1)) +
  ggplot2::labs(title = 'Make% at 50+ Yards', x = 'Season', y = NULL) +
  theme_paper()

p_d <- ggplot2::ggplot(season_trends, ggplot2::aes(season, xfg_mean)) +
  ggplot2::geom_line(colour = '#762A83', linewidth = 0.9) +
  ggplot2::geom_point(colour = '#762A83', size = 1.8) +
  ggplot2::scale_y_continuous(limits = c(0.75, 0.95), labels = scales::percent_format(accuracy = 1)) +
  ggplot2::labs(title = 'Mean xFG (M2)', x = 'Season', y = NULL) +
  theme_paper()

p_era <- (p_a | p_b) / (p_c | p_d) +
  patchwork::plot_annotation(
    title = 'The Modern Kicking Era: 2015-2025',
    subtitle = 'Broader y-axis ranges avoid overstating small percentage changes',
    theme = ggplot2::theme(
      plot.title = ggplot2::element_text(face = 'bold', size = 13),
      plot.subtitle = ggplot2::element_text(size = 9, colour = 'grey40')
    )
  )

save_fig(p_era, 'fig_07_era_comparison.png', w = 9, h = 6)

Saved: g:/Other computers/Desktop/My Files/Python/Sports Analytics Projects/Football/Kickers/NFL-Field-Goal-Kicker-Model/reports/figures/fig_07_era_comparison.png



In [14]:
# fig_08: Kicking evolution — seasonal make%, mean xFG, and mean FGOE
evolution_df <- preds_is %>%
  filter(!is.na(kick_made), !is.na(p_m2)) %>%
  group_by(season) %>%
  summarise(
    make_pct   = mean(kick_made),
    xfg_m2     = mean(p_m2),
    fgoe_mean  = mean(fgoe_m2),
    n          = n(),
    .groups = 'drop'
  )

p_evol <- ggplot2::ggplot(evolution_df, ggplot2::aes(season)) +
  ggplot2::geom_hline(yintercept = 0, colour = 'grey80', lty = 3) +
  ggplot2::geom_line(ggplot2::aes(y = make_pct, colour = 'Observed Make%'),  linewidth = 0.9) +
  ggplot2::geom_line(ggplot2::aes(y = xfg_m2,   colour = 'xFG (M2)'),        linewidth = 0.9, lty = 2) +
  ggplot2::geom_col(ggplot2::aes(y = fgoe_mean, fill = fgoe_mean > 0),
                    alpha = 0.3, width = 0.6) +
  ggplot2::scale_colour_manual(values = c('Observed Make%' = '#2166AC', 'xFG (M2)' = '#D6604D')) +
  ggplot2::scale_fill_manual(values = c('TRUE' = '#4DAC26', 'FALSE' = '#D6604D'), guide = 'none') +
  ggplot2::scale_y_continuous(labels = scales::percent_format(accuracy = 1),
                               limits = c(NA, 1)) +
  ggplot2::labs(title = 'Kicking Evolution: Observed vs. Expected Make%',
    subtitle = 'Bars show mean FGOE per season (green = over-performing expectation)',
    x = 'Season', y = NULL, colour = NULL) +
  theme_paper()

save_fig(p_evol, 'fig_08_kicking_evolution.png')


Saved: g:/Other computers/Desktop/My Files/Python/Sports Analytics Projects/Football/Kickers/NFL-Field-Goal-Kicker-Model/reports/figures/fig_08_kicking_evolution.png



In [15]:
# fig_09: FG attempt distance distribution
p_dist <- ggplot2::ggplot(preds, ggplot2::aes(kick_distance)) +
  ggplot2::geom_histogram(binwidth = 1, fill = '#2166AC', colour = NA, alpha = 0.8) +
  ggplot2::scale_x_continuous(breaks = seq(10, 70, 5)) +
  ggplot2::labs(title = 'Field Goal Attempt Distance Distribution',
    x = 'Kick Distance (yards)', y = 'Attempts') +
  theme_paper()

save_fig(p_dist, 'fig_09_distance_distribution.png')

Saved: g:/Other computers/Desktop/My Files/Python/Sports Analytics Projects/Football/Kickers/NFL-Field-Goal-Kicker-Model/reports/figures/fig_09_distance_distribution.png



In [16]:
# fig_10: FGOE leaderboard (career totals, minimum MIN_FGOE_ATT attempts)
# Uses p_m2_pop: population-level M2 WITHOUT kicker RE — fair baseline for attribution
leaderboard <- preds_is %>%
  filter(!is.na(kick_made), !is.na(p_m2_pop)) %>%
  group_by(kicker_player_id, kicker_player_name) %>%
  summarise(
    attempts   = n(),
    makes      = sum(kick_made),
    xfg        = sum(p_m2_pop),
    fgoe_total = sum(fgoe_pop),
    fgoe_per   = fgoe_total / attempts,
    seasons    = paste(min(season), max(season), sep = '-'),
    .groups    = 'drop'
  ) %>%
  filter(attempts >= MIN_FGOE_ATT) %>%
  arrange(desc(fgoe_total))

top20 <- leaderboard %>% slice_head(n = 20)

p_leader <- ggplot2::ggplot(
    top20,
    ggplot2::aes(x = fgoe_total,
                 y = reorder(kicker_player_name, fgoe_total))) +
  ggplot2::geom_col(fill = '#2166AC', alpha = 0.85) +
  ggplot2::geom_vline(xintercept = 0, colour = 'grey50') +
  ggplot2::labs(
    title    = paste0('Top 20 Kickers by Career FGOE (M2 population, min ', MIN_FGOE_ATT, ' att)'),
    subtitle = 'Expected make from population-level model (venue RE only; no kicker RE)',
    x = 'Career FGOE', y = NULL) +
  theme_paper()

save_fig(p_leader, 'fig_10_fgoe_leaderboard.png', h = 7)


Saved: g:/Other computers/Desktop/My Files/Python/Sports Analytics Projects/Football/Kickers/NFL-Field-Goal-Kicker-Model/reports/figures/fig_10_fgoe_leaderboard.png



In [17]:
# fig_11: Skill vs workload scatter (uses fgoe_pop — population-level FGOE)
p_skill <- ggplot2::ggplot(
    leaderboard,
    ggplot2::aes(attempts, fgoe_per, label = kicker_player_name)) +
  ggplot2::geom_point(ggplot2::aes(size = abs(fgoe_total)),
                       alpha = 0.6, colour = '#2166AC') +
  ggrepel::geom_text_repel(
    data = leaderboard %>% filter(
      attempts > quantile(attempts, 0.85, na.rm = TRUE) |
      abs(fgoe_per) > quantile(abs(fgoe_per), 0.90, na.rm = TRUE)
    ),
    mapping = ggplot2::aes(attempts, fgoe_per, label = kicker_player_name),
    size = 2.8, max.overlaps = 20, inherit.aes = FALSE
  ) +
  ggplot2::geom_hline(yintercept = 0, lty = 2, colour = 'grey50') +
  ggplot2::scale_size_continuous(range = c(1, 6), guide = 'none') +
  ggplot2::labs(
    title    = 'Kicker Skill vs. Workload (M2 population)',
    subtitle = 'FGOE uses population-level xFG — residual reflects individual kicker skill',
    x = 'Career FG Attempts', y = 'FGOE per Attempt') +
  theme_paper()

save_fig(p_skill, 'fig_11_skill_vs_workload.png')


Saved: g:/Other computers/Desktop/My Files/Python/Sports Analytics Projects/Football/Kickers/NFL-Field-Goal-Kicker-Model/reports/figures/fig_11_skill_vs_workload.png



## 8. Player-Level Evaluation (Consolidated)

This section groups all player-attribution visuals together using population-level M2 (`p_m2_pop` / `fgoe_pop`).

In [18]:
# fig_23: FGOE per-attempt rate leaderboard (min MIN_FGOE_ATT attempts)
rate_leader <- leaderboard %>%
  arrange(desc(fgoe_per)) %>%
  slice_head(n = 20)

p_rate_leader <- ggplot2::ggplot(
  rate_leader,
  ggplot2::aes(x = fgoe_per, y = reorder(kicker_player_name, fgoe_per))
) +
  ggplot2::geom_col(fill = '#009E73', alpha = 0.85) +
  ggplot2::geom_vline(xintercept = 0, colour = 'grey50') +
  ggplot2::scale_x_continuous(labels = scales::number_format(accuracy = 0.001)) +
  ggplot2::labs(
    title = paste0('Top 20 Kickers by FGOE per Attempt (M2 population, min ', MIN_FGOE_ATT, ' att)'),
    subtitle = 'Efficiency rate using population-level xFG (venue RE only; no kicker RE)',
    x = 'FGOE per Attempt', y = NULL
  ) +
  theme_paper()

save_fig(p_rate_leader, 'fig_23_fgoe_rate_leaderboard.png', h = 7)


Saved: g:/Other computers/Desktop/My Files/Python/Sports Analytics Projects/Football/Kickers/NFL-Field-Goal-Kicker-Model/reports/figures/fig_23_fgoe_rate_leaderboard.png



In [19]:
# fig_24: Kicker workload vs propensity context
workload_df <- preds_is %>%
  filter(!is.na(w_ipw_final), is.finite(w_ipw_final), !is.na(kick_distance)) %>%
  group_by(kicker_player_id, kicker_player_name) %>%
  summarise(
    n_attempts = n(),
    avg_distance = mean(kick_distance),
    avg_weight = mean(w_ipw_final),
    .groups = 'drop'
  ) %>%
  filter(n_attempts >= MIN_FGOE_ATT)

top_workload <- workload_df %>% arrange(desc(avg_weight)) %>% slice_head(n = 8)

p_workload <- ggplot2::ggplot(workload_df,
  ggplot2::aes(avg_distance, avg_weight, size = n_attempts, colour = avg_weight)
) +
  ggplot2::geom_point(alpha = 0.7) +
  ggrepel::geom_text_repel(
    data = top_workload,
    mapping = ggplot2::aes(x = avg_distance, y = avg_weight, label = kicker_player_name),
    size = 2.8, colour = 'grey20', max.overlaps = 20, inherit.aes = FALSE
  ) +
  ggplot2::scale_colour_gradient(low = '#1f77b4', high = '#d62728', guide = 'none') +
  ggplot2::scale_size_continuous(range = c(1.5, 7), name = 'FG Attempts') +
  ggplot2::geom_hline(yintercept = 1, lty = 2, colour = 'grey50') +
  ggplot2::labs(
    title = 'Kicker Workload: Avg Distance vs. Mean IPW Weight',
    subtitle = paste0('Colour = avg weight (red = harder context). Min ', MIN_FGOE_ATT, ' attempts.'),
    x = 'Average Kick Distance (yards)', y = 'Mean IPW Weight'
  ) +
  theme_paper()

save_fig(p_workload, 'fig_24_kicker_workload_propensity.png', w = 9, h = 6)


Saved: g:/Other computers/Desktop/My Files/Python/Sports Analytics Projects/Football/Kickers/NFL-Field-Goal-Kicker-Model/reports/figures/fig_24_kicker_workload_propensity.png



In [20]:
# fig_25 + csv: Seasonal diagnostic for Brandon Aubrey vs A. Szmyt vs J. Tucker
player_diag <- preds_is %>%
  filter(!is.na(kick_made), !is.na(fgoe_pop), !is.na(w_ipw_final)) %>%
  mutate(kicker_label = dplyr::case_when(
    stringr::str_detect(kicker_player_name, regex('aubrey', ignore_case = TRUE)) ~ 'Brandon Aubrey',
    stringr::str_detect(kicker_player_name, regex('szmyt', ignore_case = TRUE)) ~ 'Andre Szmyt',
    stringr::str_detect(kicker_player_name, regex('tucker', ignore_case = TRUE)) ~ 'Justin Tucker',
    TRUE ~ NA_character_
  )) %>%
  filter(!is.na(kicker_label)) %>%
  group_by(kicker_label, season) %>%
  summarise(
    n_kicks = n(),
    make_pct = mean(kick_made),
    season_fgoe = sum(fgoe_pop),
    mean_weight = mean(w_ipw_final),
    .groups = 'drop'
  ) %>%
  arrange(kicker_label, season)

readr::write_csv(player_diag, file.path(figures_dir, 'fig_25_player_season_diagnostic.csv'))

player_diag_long <- player_diag %>%
  tidyr::pivot_longer(
    cols = c(season_fgoe, n_kicks, make_pct, mean_weight),
    names_to = 'metric', values_to = 'value'
  ) %>%
  mutate(metric = dplyr::recode(metric,
    season_fgoe = 'Season FGOE',
    n_kicks = 'N Kicks',
    make_pct = 'Make %',
    mean_weight = 'Mean IPW Weight'
  ))

p_player_diag <- ggplot2::ggplot(
  player_diag_long,
  ggplot2::aes(season, value, colour = kicker_label)
) +
  ggplot2::geom_line(linewidth = 0.9) +
  ggplot2::geom_point(size = 1.8) +
  ggplot2::facet_wrap(~ metric, scales = 'free_y', ncol = 2) +
  ggplot2::scale_colour_manual(values = c('Brandon Aubrey' = '#D55E00', 'Andre Szmyt' = '#0072B2', 'Justin Tucker' = '#009E73')) +
  ggplot2::labs(
    title = 'Seasonal Player Diagnostic: Aubrey vs Szmyt vs Tucker',
    subtitle = 'Per-season FGOE, kick volume, make rate, and mean IPW weight',
    x = 'Season', y = NULL, colour = NULL
  ) +
  theme_paper()

save_fig(p_player_diag, 'fig_25_player_season_diagnostic.png', w = 10, h = 7)


Saved: g:/Other computers/Desktop/My Files/Python/Sports Analytics Projects/Football/Kickers/NFL-Field-Goal-Kicker-Model/reports/figures/fig_25_player_season_diagnostic.png



In [21]:
# ============================================================
# 7. Final Summary
# ============================================================
fig_files <- list.files(figures_dir, pattern = '^fig_', full.names = FALSE)
cat('\n=== Figures Generated ===\n')
for (f in sort(fig_files)) cat(' ', f, '\n')

cat('\n=== Test Set Metrics ===\n')
print(metrics)


=== Figures Generated ===
  fig_01_calibration_by_model.png 
  fig_02_residual_by_distance.png 
  fig_03_weight_distribution.png 
  fig_04_weights_vs_distance.png 
  fig_05_rationality_density.png 
  fig_06_wpa_decision_landscape.png 
  fig_07_era_comparison.png 
  fig_08_kicking_evolution.png 
  fig_09_distance_distribution.png 
  fig_10_fgoe_leaderboard.png 
  fig_11_skill_vs_workload.png 
  fig_12_metrics_table.csv 
  fig_13_truth_curve.png 
  fig_14_bias_curve.png 
  fig_15_global_density.png 
  fig_16_separation_density.png 
  fig_17_tail_density_50plus.png 
  fig_18_pessimism_gap.png 
  fig_19_distance_by_era.png 
  fig_20_unicorn_penalty.png 
  fig_21_loess_make_rate_by_weight.png 
  fig_22_m3_weight_sensitivity.png 
  fig_23_fgoe_rate_leaderboard.png 
  fig_24_kicker_workload_propensity.png 
  fig_25_player_season_diagnostic.csv 
  fig_25_player_season_diagnostic.png 
  fig_S1_calibration_test.png 
  fig_S2_brier_distance_test.png 

=== Test Set Metrics ===
# A tibble: 8 × 6
 

## 9. Distance Analysis: Truth Curves & Bias (from 08_comparison_plots)

In [22]:
# fig_13: Truth Curve — LOESS predicted probability vs distance, all models
# Build binned observed rate (±SE) as ground truth reference
truth_bins <- preds_is %>%
  filter(!is.na(kick_made), !is.na(kick_distance)) %>%
  mutate(dist_bin = round(kick_distance / 2) * 2) %>%   # 2-yard bins
  group_by(dist_bin) %>%
  summarise(obs_rate = mean(kick_made), n = n(),
            se = sqrt(obs_rate * (1 - obs_rate) / n), .groups = 'drop') %>%
  filter(n >= 5)

long_preds <- preds_is %>%
  tidyr::pivot_longer(c(p_m0, p_m1, p_m2, p_m3),
                      names_to = 'model', values_to = 'prob') %>%
  mutate(model = toupper(stringr::str_remove(model, 'p_')))

# Model colour palette (Okabe-Ito inspired)
model_cols <- c('M0' = '#525252', 'M1' = '#0072B2', 'M2' = '#D55E00', 'M3' = '#009E73')

p_truth <- ggplot2::ggplot() +
  ggplot2::geom_ribbon(data = truth_bins,
    ggplot2::aes(dist_bin, ymin = obs_rate - 1.96*se, ymax = obs_rate + 1.96*se),
    fill = 'grey85', alpha = 0.6) +
  ggplot2::geom_point(data = truth_bins,
    ggplot2::aes(dist_bin, obs_rate), colour = 'grey40', size = 1.2, shape = 16) +
  ggplot2::geom_smooth(data = long_preds,
    ggplot2::aes(kick_distance, prob, colour = model),
    method = 'loess', se = FALSE, span = 0.4, linewidth = 0.9) +
  ggplot2::scale_colour_manual(values = model_cols) +
  ggplot2::scale_y_continuous(labels = scales::percent_format(accuracy = 1),
                               limits = c(0, 1)) +
  ggplot2::labs(title = 'Truth Curve: Predicted vs. Observed Make Rate by Distance',
    subtitle = 'Grey band = observed ±1.96 SE; lines = model LOESS (in-sample)',
    x = 'Kick Distance (yards)', y = 'Make Rate / P(make)', colour = NULL) +
  theme_paper()

save_fig(p_truth, 'fig_13_truth_curve.png', w = 9, h = 5.5)


`geom_smooth()` using formula = 'y ~ x'
Warning message:
"Removed 2 rows containing missing values or values outside the scale range
(`geom_ribbon()`)."
Warning message:
"Removed 16 rows containing missing values or values outside the scale range
(`geom_smooth()`)."
Saved: g:/Other computers/Desktop/My Files/Python/Sports Analytics Projects/Football/Kickers/NFL-Field-Goal-Kicker-Model/reports/figures/fig_13_truth_curve.png



In [23]:
# fig_14: Bias Curve — signed residual (predicted − observed) by distance, LOESS
bias_long <- preds_is %>%
  filter(!is.na(kick_made)) %>%
  tidyr::pivot_longer(c(p_m0, p_m1, p_m2, p_m3),
                      names_to = 'model', values_to = 'prob') %>%
  mutate(model    = toupper(stringr::str_remove(model, 'p_')),
         residual = prob - kick_made)

p_bias <- ggplot2::ggplot(bias_long,
    ggplot2::aes(kick_distance, residual, colour = model)) +
  ggplot2::geom_hline(yintercept = 0, lty = 2, colour = 'grey50') +
  ggplot2::geom_smooth(method = 'loess', se = TRUE, alpha = 0.08,
                        span = 0.4, linewidth = 0.9) +
  ggplot2::scale_colour_manual(values = model_cols) +
  ggplot2::labs(title = 'Bias Curve: Model Residual by Distance (In-Sample)',
    subtitle = 'Positive = overestimates make probability; negative = underestimates',
    x = 'Kick Distance (yards)', y = 'P(make) \u2212 Observed', colour = NULL) +
  theme_paper()

save_fig(p_bias, 'fig_14_bias_curve.png', w = 9, h = 5.5)


`geom_smooth()` using formula = 'y ~ x'
Saved: g:/Other computers/Desktop/My Files/Python/Sports Analytics Projects/Football/Kickers/NFL-Field-Goal-Kicker-Model/reports/figures/fig_14_bias_curve.png



In [24]:
# fig_15 + fig_16: Probability Distributions — global density and separation by outcome
dens_long <- preds_is %>%
  filter(!is.na(kick_made)) %>%
  tidyr::pivot_longer(c(p_m1, p_m2, p_m3),
                      names_to = 'model', values_to = 'prob') %>%
  mutate(model   = toupper(stringr::str_remove(model, 'p_')),
         outcome = if_else(kick_made == 1L, 'Made', 'Missed'))

model_cols_3 <- c('M1' = '#0072B2', 'M2' = '#D55E00', 'M3' = '#009E73')

# fig_15: Global density of all predicted probabilities
p_dens_global <- ggplot2::ggplot(dens_long,
    ggplot2::aes(prob, colour = model, fill = model)) +
  ggplot2::geom_density(alpha = 0.15, linewidth = 0.8) +
  ggplot2::scale_colour_manual(values = model_cols_3) +
  ggplot2::scale_fill_manual(values = model_cols_3) +
  ggplot2::labs(title = 'Global Density of Predicted Probabilities (In-Sample)',
    x = 'P(make)', y = 'Density', colour = NULL, fill = NULL) +
  theme_paper()

save_fig(p_dens_global, 'fig_15_global_density.png')

# fig_16: Separation — density faceted by outcome (Made / Missed)
p_dens_sep <- ggplot2::ggplot(dens_long,
    ggplot2::aes(prob, colour = model, fill = model)) +
  ggplot2::geom_density(alpha = 0.2, linewidth = 0.8) +
  ggplot2::facet_wrap(~ outcome, ncol = 2) +
  ggplot2::scale_colour_manual(values = model_cols_3) +
  ggplot2::scale_fill_manual(values = model_cols_3) +
  ggplot2::labs(title = 'Probability Separation: Made vs. Missed (In-Sample)',
    subtitle = 'Models should assign higher probabilities to made kicks',
    x = 'P(make)', y = 'Density', colour = NULL, fill = NULL) +
  theme_paper()

save_fig(p_dens_sep, 'fig_16_separation_density.png')


Saved: g:/Other computers/Desktop/My Files/Python/Sports Analytics Projects/Football/Kickers/NFL-Field-Goal-Kicker-Model/reports/figures/fig_15_global_density.png

Saved: g:/Other computers/Desktop/My Files/Python/Sports Analytics Projects/Football/Kickers/NFL-Field-Goal-Kicker-Model/reports/figures/fig_16_separation_density.png



In [25]:
# fig_17: Tail Analysis — density of predicted probabilities for 50+ yard kicks
tail_long <- preds_is %>%
  filter(!is.na(kick_made), kick_distance >= 50) %>%
  tidyr::pivot_longer(c(p_m1, p_m2, p_m3),
                      names_to = 'model', values_to = 'prob') %>%
  mutate(model = toupper(stringr::str_remove(model, 'p_')))

n_tail <- nrow(preds_is %>% filter(kick_distance >= 50))

p_tail <- ggplot2::ggplot(tail_long,
    ggplot2::aes(prob, colour = model, fill = model)) +
  ggplot2::geom_density(alpha = 0.2, linewidth = 0.8) +
  ggplot2::geom_vline(xintercept = 0.5, lty = 2, colour = 'grey40') +
  ggplot2::scale_colour_manual(values = model_cols_3) +
  ggplot2::scale_fill_manual(values = model_cols_3) +
  ggplot2::labs(
    title   = paste0('Long-Distance Kick Probabilities: 50+ Yards (n = ', n_tail, ', in-sample)'),
    subtitle = 'Dashed line = 50% threshold; rightward shift = more optimistic long-range estimate',
    x = 'P(make)', y = 'Density', colour = NULL, fill = NULL) +
  theme_paper()

save_fig(p_tail, 'fig_17_tail_density_50plus.png')


Saved: g:/Other computers/Desktop/My Files/Python/Sports Analytics Projects/Football/Kickers/NFL-Field-Goal-Kicker-Model/reports/figures/fig_17_tail_density_50plus.png



In [26]:
# fig_18: Pessimism Gap — signed mean bias per distance band, bar chart
pessimism_df <- preds_is %>%
  filter(!is.na(kick_made)) %>%
  mutate(dist_band = cut(kick_distance,
    breaks = c(-Inf, 25, 30, 35, 40, 45, 50, 55, 60, Inf),
    labels = c('<25','26-30','31-35','36-40','41-45','46-50','51-55','56-60','>60'),
    include.lowest = TRUE)) %>%
  tidyr::pivot_longer(c(p_m1, p_m2, p_m3),
                      names_to = 'model', values_to = 'prob') %>%
  mutate(model    = toupper(stringr::str_remove(model, 'p_')),
         residual = prob - kick_made) %>%
  group_by(model, dist_band) %>%
  summarise(bias = mean(residual), n = n(), .groups = 'drop')

p_pessimism <- ggplot2::ggplot(pessimism_df,
    ggplot2::aes(dist_band, bias, fill = model)) +
  ggplot2::geom_col(position = 'dodge', alpha = 0.85, width = 0.75) +
  ggplot2::geom_hline(yintercept = 0, colour = 'grey40') +
  ggplot2::scale_fill_manual(values = model_cols_3) +
  ggplot2::scale_y_continuous(labels = scales::number_format(accuracy = 0.01)) +
  ggplot2::labs(title = 'Pessimism Gap: Signed Bias by Distance Band (In-Sample)',
    subtitle = 'Positive = over-predicts make probability; negative = under-predicts',
    x = 'Distance Band (yards)', y = 'Mean Bias', fill = NULL) +
  theme_paper() +
  ggplot2::theme(axis.text.x = ggplot2::element_text(angle = 30, hjust = 1))

save_fig(p_pessimism, 'fig_18_pessimism_gap.png')


Saved: g:/Other computers/Desktop/My Files/Python/Sports Analytics Projects/Football/Kickers/NFL-Field-Goal-Kicker-Model/reports/figures/fig_18_pessimism_gap.png



## 10. Era Trend: Distance Distribution Shift 2015-2025

In [27]:
# fig_19: Distance distribution by era — overlaid density curves on one axis
era_labels <- c('2015-2018', '2019-2022', '2023-2025')

dist_era <- preds_is %>%
  filter(!is.na(kick_distance)) %>%
  mutate(era_grp = dplyr::case_when(
    season <= 2018 ~ era_labels[1],
    season <= 2022 ~ era_labels[2],
    TRUE           ~ era_labels[3]
  ) %>% factor(levels = era_labels))

p_dist_era <- ggplot2::ggplot(dist_era, ggplot2::aes(kick_distance, colour = era_grp, fill = era_grp)) +
  ggplot2::geom_density(alpha = 0.12, linewidth = 1.0, adjust = 1.0) +
  ggplot2::geom_vline(xintercept = 50, lty = 2, colour = 'grey35') +
  ggplot2::scale_colour_manual(values = c('2015-2018' = '#A6D96A', '2019-2022' = '#66BD63', '2023-2025' = '#1A9850')) +
  ggplot2::scale_fill_manual(values = c('2015-2018' = '#A6D96A', '2019-2022' = '#66BD63', '2023-2025' = '#1A9850')) +
  ggplot2::scale_x_continuous(breaks = seq(10, 70, 5)) +
  ggplot2::labs(
    title = 'FG Attempt Distance Distribution by Era (2015-2025)',
    subtitle = 'Three overlaid densities on a common axis; dashed line = 50 yards',
    x = 'Kick Distance (yards)', y = 'Density',
    colour = 'Era', fill = 'Era'
  ) +
  theme_paper()

save_fig(p_dist_era, 'fig_19_distance_by_era.png', w = 8, h = 5.8)

Saved: g:/Other computers/Desktop/My Files/Python/Sports Analytics Projects/Football/Kickers/NFL-Field-Goal-Kicker-Model/reports/figures/fig_19_distance_by_era.png



## 11. IPW Weight Diagnostics: Unicorn Penalty

In [28]:
# fig_20: Unicorn Penalty Bars — make rate by distance bin, split by weight class
# High-weight kicks (w > 1) are harder-than-average; this confirms IPW is capturing that
unicorn_df <- preds_is %>%
  filter(!is.na(kick_made), !is.na(w_ipw_final), is.finite(w_ipw_final),
         kick_distance >= 37, kick_distance <= 65) %>%
  mutate(
    dist_bin    = cut(kick_distance,
      breaks = seq(35, 65, by = 5),
      labels = c('36-40','41-45','46-50','51-55','56-60','61-65'),
      include.lowest = TRUE),
    weight_class = if_else(w_ipw_final > 1, 'High weight (>1)', 'Low weight (\u22641)')
  ) %>%
  filter(!is.na(dist_bin)) %>%
  group_by(dist_bin, weight_class) %>%
  summarise(make_rate = mean(kick_made), n = n(),
            se = sqrt(make_rate*(1-make_rate)/n), .groups = 'drop')

p_unicorn <- ggplot2::ggplot(unicorn_df,
    ggplot2::aes(dist_bin, make_rate, fill = weight_class)) +
  ggplot2::geom_col(position = 'dodge', alpha = 0.85, width = 0.7) +
  ggplot2::geom_errorbar(
    ggplot2::aes(ymin = make_rate - 1.96*se, ymax = make_rate + 1.96*se),
    position = ggplot2::position_dodge(width = 0.7), width = 0.2) +
  ggplot2::geom_text(
    ggplot2::aes(label = n, y = 0.02),
    position = ggplot2::position_dodge(width = 0.7),
    size = 2.6, colour = 'white', fontface = 'bold') +
  ggplot2::scale_fill_manual(values = c('Low weight (\u22641)' = '#2166AC',
                                         'High weight (>1)'      = '#D6604D')) +
  ggplot2::scale_y_continuous(labels = scales::percent_format(accuracy = 1),
                               limits = c(0, 1)) +
  ggplot2::labs(
    title    = 'The "Unicorn Penalty": Make Rate by Weight Class and Distance',
    subtitle = 'High-weight kicks are harder-than-expected conditional on distance (n shown in bar)',
    x = 'Kick Distance (yards)', y = 'Observed Make Rate', fill = NULL) +
  theme_paper()

save_fig(p_unicorn, 'fig_20_unicorn_penalty.png')


Saved: g:/Other computers/Desktop/My Files/Python/Sports Analytics Projects/Football/Kickers/NFL-Field-Goal-Kicker-Model/reports/figures/fig_20_unicorn_penalty.png



In [29]:
# fig_21: Continuous LOESS make rate by weight class over full distance range
loess_weight_df <- preds_is %>%
  filter(!is.na(kick_made), !is.na(w_ipw_final), is.finite(w_ipw_final)) %>%
  mutate(weight_class = if_else(w_ipw_final > 1, 'High weight (>1)', 'Low weight (\u22641)'))

p_loess_w <- ggplot2::ggplot(loess_weight_df,
    ggplot2::aes(kick_distance, kick_made, colour = weight_class)) +
  ggplot2::geom_smooth(method = 'loess', span = 0.7, se = TRUE, alpha = 0.12,
                        linewidth = 1.0) +
  ggplot2::scale_colour_manual(values = c('Low weight (\u22641)' = '#2166AC',
                                           'High weight (>1)'      = '#D6604D')) +
  ggplot2::scale_y_continuous(labels = scales::percent_format(accuracy = 1)) +
  ggplot2::labs(
    title    = 'Observed Make Rate by Weight Class: LOESS (In-Sample)',
    subtitle = 'High-weight kicks consistently harder than distance alone predicts',
    x = 'Kick Distance (yards)', y = 'Observed Make Rate', colour = NULL) +
  theme_paper()

save_fig(p_loess_w, 'fig_21_loess_make_rate_by_weight.png')


`geom_smooth()` using formula = 'y ~ x'
Saved: g:/Other computers/Desktop/My Files/Python/Sports Analytics Projects/Football/Kickers/NFL-Field-Goal-Kicker-Model/reports/figures/fig_21_loess_make_rate_by_weight.png



## 12. M3 Augmentation Weight Sensitivity

In [30]:
# fig_22: M3 augmentation weight sensitivity — AUC/Brier + coefficient shift across variants
grid_df <- readr::read_csv(
  file.path(reports_dir, 'xfg_success', 'm3_weight_grid.csv'),
  show_col_types = FALSE
) %>%
  mutate(weight_label = paste0('w = ', pat_weight))

# Panel A: AUC and Brier IS across weight variants
grid_metrics <- grid_df %>%
  tidyr::pivot_longer(c(auc_is_fg, brier_is_fg),
                      names_to = 'metric', values_to = 'value') %>%
  mutate(metric = dplyr::recode(metric,
    'auc_is_fg'   = 'AUC (IS, FG only)',
    'brier_is_fg' = 'Brier (IS, FG only)'
  ))

p_grid_metrics <- ggplot2::ggplot(grid_metrics,
    ggplot2::aes(weight_label, value, group = metric, colour = metric)) +
  ggplot2::geom_line(linewidth = 0.9) +
  ggplot2::geom_point(size = 3) +
  ggplot2::facet_wrap(~ metric, scales = 'free_y', ncol = 2) +
  ggplot2::scale_colour_manual(
    values = c('AUC (IS, FG only)' = '#2166AC', 'Brier (IS, FG only)' = '#D6604D'),
    guide = 'none') +
  ggplot2::labs(title = 'M3 Augmentation Weight Sensitivity: IS FG Metrics',
    x = 'PAT = NON weight', y = NULL) +
  theme_paper()

# Panel B: Coefficient shift (intercept and dist_bs_6)
grid_coefs <- grid_df %>%
  tidyr::pivot_longer(c(coef_intercept, coef_dist_bs_6),
                      names_to = 'coef', values_to = 'value') %>%
  mutate(coef = dplyr::recode(coef,
    'coef_intercept'  = 'Intercept',
    'coef_dist_bs_6'  = 'dist_bs_6 (long range)'
  ))

p_grid_coefs <- ggplot2::ggplot(grid_coefs,
    ggplot2::aes(weight_label, value, group = coef, colour = coef)) +
  ggplot2::geom_line(linewidth = 0.9) +
  ggplot2::geom_point(size = 3) +
  ggplot2::facet_wrap(~ coef, scales = 'free_y', ncol = 2) +
  ggplot2::scale_colour_manual(
    values = c('Intercept' = '#2166AC', 'dist_bs_6 (long range)' = '#009E73'),
    guide = 'none') +
  ggplot2::labs(title = 'M3 Coefficient Shift Across Weight Variants',
    x = 'PAT = NON weight', y = 'Coefficient value') +
  theme_paper()

p_m3_grid <- p_grid_metrics / p_grid_coefs +
  patchwork::plot_annotation(
    title    = 'M3 Augmentation Sensitivity: PAT = NON weight \u2208 {0.05, 0.10, 0.25}',
    subtitle = 'Metrics are FG-only in-sample; model is robust across this weight range',
    theme = ggplot2::theme(
      plot.title    = ggplot2::element_text(face = 'bold', size = 12),
      plot.subtitle = ggplot2::element_text(size = 9, colour = 'grey40')
    )
  )

save_fig(p_m3_grid, 'fig_22_m3_weight_sensitivity.png', w = 9, h = 7)


Saved: g:/Other computers/Desktop/My Files/Python/Sports Analytics Projects/Football/Kickers/NFL-Field-Goal-Kicker-Model/reports/figures/fig_22_m3_weight_sensitivity.png



## 13. Supplemental: Test-Set Metrics

The following figures repeat the core calibration and Brier diagnostics on the held-out test set for reference. All primary analysis above uses in-sample (2015-2025) results.

In [31]:
# fig_S1: Calibration curves — test set
calib_df_test <- dplyr::bind_rows(
  calib_data(y_test, preds_test$p_m0, label = 'M0'),
  calib_data(y_test, preds_test$p_m1, label = 'M1'),
  calib_data(y_test, preds_test$p_m2, label = 'M2'),
  calib_data(y_test, preds_test$p_m3, label = 'M3')
)

p_calib_test <- ggplot2::ggplot(calib_df_test, ggplot2::aes(pred, obs, colour = model)) +
  ggplot2::geom_abline(slope = 1, intercept = 0, lty = 2, colour = 'grey50') +
  ggplot2::geom_line(linewidth = 0.9) +
  ggplot2::geom_point(ggplot2::aes(size = n)) +
  ggplot2::scale_size_continuous(range = c(1.5, 5), guide = 'none') +
  ggplot2::scale_colour_brewer(palette = 'Set1') +
  ggplot2::coord_equal(xlim = c(0, 1), ylim = c(0, 1)) +
  ggplot2::labs(title = paste0('Calibration Curves (Test Set, n = ', nrow(preds_test), ')'),
    x = 'Mean Predicted P(make)', y = 'Observed Make Rate', colour = NULL) +
  theme_paper()

save_fig(p_calib_test, 'fig_S1_calibration_test.png')

# fig_S2: Brier by distance — test set
resid_df_test <- preds_test %>%
  filter(!is.na(kick_made)) %>%
  mutate(dist_bin = cut(kick_distance,
    breaks = c(-Inf, 25, 30, 35, 40, 45, 50, 55, 60, Inf),
    labels = c('<25','26-30','31-35','36-40','41-45','46-50','51-55','56-60','>60'),
    include.lowest = TRUE)) %>%
  tidyr::pivot_longer(c(p_m0, p_m1, p_m2, p_m3),
                      names_to = 'model', values_to = 'p') %>%
  mutate(model = toupper(stringr::str_remove(model, 'p_'))) %>%
  group_by(model, dist_bin) %>%
  summarise(brier_bin = brier(kick_made, p), n = n(), .groups = 'drop')

p_resid_test <- ggplot2::ggplot(resid_df_test,
    ggplot2::aes(dist_bin, brier_bin, group = model, colour = model)) +
  ggplot2::geom_line(linewidth = 0.9) +
  ggplot2::geom_point() +
  ggplot2::scale_colour_brewer(palette = 'Set1') +
  ggplot2::labs(title = paste0('Brier Score by Distance Band (Test Set, n = ', nrow(preds_test), ')'),
    x = 'Kick Distance (yards)', y = 'Brier Score', colour = NULL) +
  theme_paper() +
  ggplot2::theme(axis.text.x = ggplot2::element_text(angle = 30, hjust = 1))

save_fig(p_resid_test, 'fig_S2_brier_distance_test.png')

cat('\n=== Test Set Metrics (Supplemental) ===\n')
print(metrics_test %>% select(-split))

Saved: g:/Other computers/Desktop/My Files/Python/Sports Analytics Projects/Football/Kickers/NFL-Field-Goal-Kicker-Model/reports/figures/fig_S1_calibration_test.png

Saved: g:/Other computers/Desktop/My Files/Python/Sports Analytics Projects/Football/Kickers/NFL-Field-Goal-Kicker-Model/reports/figures/fig_S2_brier_distance_test.png




=== Test Set Metrics (Supplemental) ===
# A tibble: 4 × 5
  model                  n brier logloss   auc
  <chr>              <int> <dbl>   <dbl> <dbl>
1 M0 (distance only)  2251 0.101   0.332 0.762
2 M1 (full GLMM)      2251 0.101   0.330 0.769
3 M2 (IPW-corrected)  2251 0.103   0.337 0.760
4 M3 (augmented)      2251 0.102   0.333 0.767


In [32]:
# ============================================================
# 14. Final Summary
# ============================================================
fig_files <- list.files(figures_dir, pattern = '^fig_', full.names = FALSE)
cat('\n=== Figures Generated ===\n')
for (f in sort(fig_files)) cat(' ', f, '\n')

cat('\n=== In-Sample Metrics (Primary) ===\n')
print(metrics_is %>% select(-split))

cat('\n=== Test Metrics (Supplemental) ===\n')
print(metrics_test %>% select(-split))



=== Figures Generated ===
  fig_01_calibration_by_model.png 
  fig_02_residual_by_distance.png 
  fig_03_weight_distribution.png 
  fig_04_weights_vs_distance.png 
  fig_05_rationality_density.png 
  fig_06_wpa_decision_landscape.png 
  fig_07_era_comparison.png 
  fig_08_kicking_evolution.png 
  fig_09_distance_distribution.png 
  fig_10_fgoe_leaderboard.png 
  fig_11_skill_vs_workload.png 
  fig_12_metrics_table.csv 
  fig_13_truth_curve.png 
  fig_14_bias_curve.png 
  fig_15_global_density.png 
  fig_16_separation_density.png 
  fig_17_tail_density_50plus.png 
  fig_18_pessimism_gap.png 
  fig_19_distance_by_era.png 
  fig_20_unicorn_penalty.png 
  fig_21_loess_make_rate_by_weight.png 
  fig_22_m3_weight_sensitivity.png 
  fig_23_fgoe_rate_leaderboard.png 
  fig_24_kicker_workload_propensity.png 
  fig_25_player_season_diagnostic.csv 
  fig_25_player_season_diagnostic.png 
  fig_S1_calibration_test.png 
  fig_S2_brier_distance_test.png 

=== In-Sample Metrics (Primary) ===
# A tibb